In [ ]:
%pip install -q transformers scikit-learn seaborn matplotlib pillow tqdm

from pathlib import Path
import json, math, random, time
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import CLIPModel, CLIPTokenizerFast, CLIPImageProcessor, get_cosine_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


In [ ]:
%pip install -q transformers scikit-learn seaborn matplotlib pillow tqdm

from pathlib import Path
import os
import sys
import json
import math
import random
import time

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import (
    CLIPModel,
    CLIPTokenizerFast,
    CLIPImageProcessor,
    get_cosine_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt
import seaborn as sns

os.environ["TOKENIZERS_PARALLELISM"] = "false"

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Imports OK")
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    print("CUDA not available")


In [ ]:
ROOT = Path("/teamspace/studios/this_studio/Fakeddit")

TRAIN_CSV = ROOT / "subsets/train_100k_cached.csv"
VAL_CSV = ROOT / "subsets/validate_10k_cached.csv"

TRAIN_IMG_DIR = ROOT / "images/train"
VAL_IMG_DIR = ROOT / "images/validate"

OUT_DIR = ROOT / "outputs/cmcn_no_ocr_clip_l14"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "openai/clip-vit-large-patch14"
NUM_CLASSES = 6
CLASS_NAMES = ["True", "Satire", "Misleading", "False", "Manipulated", "Connection"]

BATCH_SIZE = 32
GRAD_ACCUM = 1
NUM_WORKERS = 8

EPOCHS_STAGE1 = 3
EPOCHS_STAGE2 = 2

LR_HEAD = 2e-4
LR_CLIP = 1e-6
WEIGHT_DECAY = 1e-4
MAX_LEN = 77

LABEL_COL = "6_way_label"
TRUE_LABEL_ID = 0

print("Checking paths...")
print("ROOT:", ROOT, ROOT.exists())
print("TRAIN_CSV:", TRAIN_CSV, TRAIN_CSV.exists())
print("VAL_CSV:", VAL_CSV, VAL_CSV.exists())
print("TRAIN_IMG_DIR:", TRAIN_IMG_DIR, TRAIN_IMG_DIR.exists())
print("VAL_IMG_DIR:", VAL_IMG_DIR, VAL_IMG_DIR.exists())
print("OUT_DIR:", OUT_DIR, OUT_DIR.exists())

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

def resolve_text_col(df):
    candidates = ["clean_title", "title", "clean_text", "text", "body"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"No text column found. Columns are: {list(df.columns)}")

TEXT_COL = resolve_text_col(train_df)

def attach_image_paths(df, image_dir):
    df = df.copy()

    if "submission_id" not in df.columns:
        if "id" in df.columns:
            df = df.rename(columns={"id": "submission_id"})
        else:
            raise ValueError("CSV needs submission_id or id column")

    df["submission_id"] = df["submission_id"].astype(str)
    df["image_path"] = df["submission_id"].apply(lambda x: str(image_dir / f"{x}.jpg"))
    df["image_exists"] = df["image_path"].apply(lambda p: Path(p).exists())
    return df

train_df = attach_image_paths(train_df, TRAIN_IMG_DIR)
val_df = attach_image_paths(val_df, VAL_IMG_DIR)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nText column:", TEXT_COL)
print("Label column:", LABEL_COL)

print("\ntrain:", train_df.shape)
print(train_df[LABEL_COL].value_counts().sort_index())
print("train image exists:")
print(train_df["image_exists"].value_counts(dropna=False))

print("\nval:", val_df.shape)
print(val_df[LABEL_COL].value_counts().sort_index())
print("val image exists:")
print(val_df["image_exists"].value_counts(dropna=False))

assert train_df["image_exists"].all(), "Some train images are missing"
assert val_df["image_exists"].all(), "Some val images are missing"
assert set(train_df[LABEL_COL].unique()).issubset(set(range(NUM_CLASSES)))
assert set(val_df[LABEL_COL].unique()).issubset(set(range(NUM_CLASSES)))

display(train_df[["submission_id", TEXT_COL, LABEL_COL, "image_path"]].head())


In [ ]:
print("Loading CLIP preprocessing...")
tokenizer = CLIPTokenizerFast.from_pretrained(MODEL_NAME)
image_processor = CLIPImageProcessor.from_pretrained(MODEL_NAME)

def collate(batch):
    return {
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": torch.stack([x["labels"] for x in batch]),
        "submission_id": [x["submission_id"] for x in batch],
        "text": [x["text"] for x in batch],
        "image_path": [x["image_path"] for x in batch],
    }

counts = train_df[LABEL_COL].value_counts().to_dict()
sample_weights = train_df[LABEL_COL].map(lambda y: 1.0 / counts[int(y)]).values

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

class_counts = train_df[LABEL_COL].value_counts().reindex(range(NUM_CLASSES), fill_value=1).sort_index()
class_weights_np = len(train_df) / (NUM_CLASSES * class_counts.values)
class_weights_np = class_weights_np / class_weights_np.mean()
class_weights = torch.tensor(class_weights_np, dtype=torch.float32).to(device)

print("Preprocessing/sampler/class_weights ready")
print("Class weights:", class_weights.detach().cpu().numpy())


In [ ]:
BAD_IMAGE_LOG = OUT_DIR / "bad_images_seen.txt"

class FakedditDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def _safe_open_image(self, image_path):
        try:
            return Image.open(image_path).convert("RGB")
        except Exception:
            # Log corrupted/unreadable images and use a blank placeholder.
            with open(BAD_IMAGE_LOG, "a") as f:
                f.write(str(image_path) + "\n")
            return Image.new("RGB", (224, 224), color=(0, 0, 0))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = str(row.get(TEXT_COL, "") or "").strip()
        if not text:
            text = "[empty]"

        enc = tokenizer(
            text,
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        image = self._safe_open_image(row["image_path"])
        pixels = image_processor(images=image, return_tensors="pt")["pixel_values"][0]

        return {
            "input_ids": enc["input_ids"][0],
            "attention_mask": enc["attention_mask"][0],
            "pixel_values": pixels,
            "labels": torch.tensor(int(row[LABEL_COL]), dtype=torch.long),
            "submission_id": row["submission_id"],
            "text": text,
            "image_path": row["image_path"],
        }


train_dataset = FakedditDataset(train_df)
val_dataset = FakedditDataset(val_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate,
    persistent_workers=NUM_WORKERS > 0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate,
    persistent_workers=NUM_WORKERS > 0,
)

print("Patched dataset with safe image loading")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Bad image log:", BAD_IMAGE_LOG)


In [ ]:
class CMCN(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_classes=NUM_CLASSES):
        super().__init__()

        self.clip = CLIPModel.from_pretrained(model_name)

        self.embed_dim = self.clip.config.projection_dim
        self.text_hidden_dim = self.clip.config.text_config.hidden_size
        self.vision_hidden_dim = self.clip.config.vision_config.hidden_size

        # Project token-level hidden states to CLIP projection dimension.
        self.text_seq_proj = nn.Linear(self.text_hidden_dim, self.embed_dim)
        self.image_seq_proj = nn.Linear(self.vision_hidden_dim, self.embed_dim)

        # Cross-modal attention: text tokens query image patch tokens.
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=self.embed_dim,
            num_heads=8,
            dropout=0.10,
            batch_first=True,
        )
        self.cross_norm = nn.LayerNorm(self.embed_dim)

        # Unimodal prediction heads for ambiguity estimation.
        self.text_head = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Linear(self.embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, num_classes),
        )

        self.image_head = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Linear(self.embed_dim, 256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, num_classes),
        )

        # Adaptive fusion gate from [similarity, ambiguity].
        self.gate = nn.Sequential(
            nn.Linear(2, 32),
            nn.GELU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

        # F_final = [F_gated | F_cross | F_text | sim | ambiguity]
        final_dim = self.embed_dim * 3 + 2

        self.classifier = nn.Sequential(
            nn.Linear(final_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.30),

            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.20),

            nn.Linear(256, num_classes),
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        out = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_dict=True,
        )

        f_text = F.normalize(out.text_embeds, dim=-1)
        f_image = F.normalize(out.image_embeds, dim=-1)

        # Branch A: cosine similarity.
        sim_score = F.cosine_similarity(f_text, f_image, dim=-1).unsqueeze(1)

        # Branch C: ambiguity from unimodal disagreement.
        text_logits = self.text_head(f_text)
        image_logits = self.image_head(f_image)

        p_text = F.softmax(text_logits, dim=-1)
        p_image = F.softmax(image_logits, dim=-1)

        ambiguity = 0.5 * (
            F.kl_div(
                F.log_softmax(text_logits, dim=-1),
                p_image,
                reduction="none",
            ).sum(dim=1, keepdim=True)
            +
            F.kl_div(
                F.log_softmax(image_logits, dim=-1),
                p_text,
                reduction="none",
            ).sum(dim=1, keepdim=True)
        )

        # Branch B: cross-modal attention.
        text_tokens = self.text_seq_proj(out.text_model_output.last_hidden_state)
        image_tokens = self.image_seq_proj(out.vision_model_output.last_hidden_state[:, 1:, :])

        cross_tokens, _ = self.cross_attn(
            query=text_tokens,
            key=image_tokens,
            value=image_tokens,
            need_weights=False,
        )

        mask = attention_mask.unsqueeze(-1).to(cross_tokens.dtype)
        f_cross = (cross_tokens * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
        f_cross = self.cross_norm(f_cross)

        # Adaptive gated fusion.
        alpha = self.gate(torch.cat([sim_score, ambiguity], dim=1))
        f_gated = alpha * f_text + (1.0 - alpha) * f_image

        f_final = torch.cat(
            [
                f_gated,
                f_cross,
                f_text,
                sim_score,
                ambiguity,
            ],
            dim=1,
        )

        logits = self.classifier(f_final)

        return {
            "logits": logits,
            "f_text": f_text,
            "f_image": f_image,
            "f_cross": f_cross,
            "sim_score": sim_score.squeeze(1),
            "ambiguity": ambiguity.squeeze(1),
            "alpha": alpha.squeeze(1),
        }


model = CMCN().to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model loaded")
print("CLIP model:", MODEL_NAME)
print("Embedding dim:", model.embed_dim)
print("Final feature dim:", model.embed_dim * 3 + 2)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters before freezing:", f"{trainable_params:,}")


In [ ]:
def set_clip_trainable(unfreeze_last_layers=0, train_clip_projection=False):
    # Freeze all CLIP parameters first.
    for p in model.clip.parameters():
        p.requires_grad = False

    # Optional fine-tuning of last text/vision transformer layers.
    if unfreeze_last_layers > 0:
        for layer in model.clip.text_model.encoder.layers[-unfreeze_last_layers:]:
            for p in layer.parameters():
                p.requires_grad = True

        for layer in model.clip.vision_model.encoder.layers[-unfreeze_last_layers:]:
            for p in layer.parameters():
                p.requires_grad = True

        if train_clip_projection:
            for p in model.clip.text_projection.parameters():
                p.requires_grad = True
            for p in model.clip.visual_projection.parameters():
                p.requires_grad = True


def count_params():
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def cmcn_aux_losses(outputs, labels, temperature=0.07, fake_margin=0.20):
    """
    Contrastive:
    - real pairs: pull matched text/image embeddings together with InfoNCE
    - fake/non-true pairs: penalize high similarity above fake_margin

    Consistency:
    - similarity should be high for True class and low for non-True classes
    """
    f_text = outputs["f_text"]
    f_image = outputs["f_image"]
    sim = outputs["sim_score"]

    real_mask = labels == TRUE_LABEL_ID
    fake_mask = ~real_mask

    zero = sim.new_tensor(0.0)
    contrastive_loss = zero

    if int(real_mask.sum()) >= 2:
        t = f_text[real_mask]
        v = f_image[real_mask]

        logits = (t @ v.T) / temperature
        targets = torch.arange(logits.size(0), device=logits.device)

        contrastive_loss = 0.5 * (
            F.cross_entropy(logits, targets)
            +
            F.cross_entropy(logits.T, targets)
        )

    if int(fake_mask.sum()) > 0:
        fake_loss = F.relu(sim[fake_mask] - fake_margin).pow(2).mean()
        contrastive_loss = contrastive_loss + fake_loss

    sim_01 = (sim + 1.0) / 2.0
    consistency_target = real_mask.float()
    consistency_loss = F.mse_loss(sim_01, consistency_target)

    return contrastive_loss, consistency_loss

def compute_loss(outputs, labels, use_aux=False):
    # Match class weight dtype to logits dtype under bf16 autocast.
    weight = class_weights.to(
        device=outputs["logits"].device,
        dtype=outputs["logits"].dtype,
    )

    cls_loss = F.cross_entropy(
        outputs["logits"],
        labels,
        weight=weight,
    )

    contrastive_loss, consistency_loss = cmcn_aux_losses(outputs, labels)

    if use_aux:
        total_loss = cls_loss + 0.3 * contrastive_loss + 0.1 * consistency_loss
    else:
        total_loss = cls_loss

    logs = {
        "loss": float(total_loss.detach().float().cpu()),
        "cls_loss": float(cls_loss.detach().float().cpu()),
        "contrastive_loss": float(contrastive_loss.detach().float().cpu()),
        "consistency_loss": float(consistency_loss.detach().float().cpu()),
    }

    return total_loss, logs

print("compute_loss patched for bf16")



def make_optimizer_and_scheduler(epochs, head_lr=LR_HEAD, clip_lr=LR_CLIP):
    head_params = []
    clip_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue

        if name.startswith("clip."):
            clip_params.append(p)
        else:
            head_params.append(p)

    param_groups = []

    if head_params:
        param_groups.append({
            "params": head_params,
            "lr": head_lr,
            "weight_decay": WEIGHT_DECAY,
        })

    if clip_params:
        param_groups.append({
            "params": clip_params,
            "lr": clip_lr,
            "weight_decay": WEIGHT_DECAY,
        })

    optimizer = torch.optim.AdamW(param_groups)

    update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
    total_steps = max(1, update_steps_per_epoch * epochs)
    warmup_steps = int(0.06 * total_steps)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    return optimizer, scheduler


def move_batch_to_device(batch):
    return {
        "input_ids": batch["input_ids"].to(device, non_blocking=True),
        "attention_mask": batch["attention_mask"].to(device, non_blocking=True),
        "pixel_values": batch["pixel_values"].to(device, non_blocking=True),
        "labels": batch["labels"].to(device, non_blocking=True),
    }


def amp_context():
    return torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
        enabled=(device.type == "cuda" and torch.cuda.is_bf16_supported()),
    )


# Stage 1 default: freeze CLIP completely.
set_clip_trainable(unfreeze_last_layers=0)

total, trainable = count_params()
print("After freezing CLIP")
print("Total parameters:", f"{total:,}")
print("Trainable parameters:", f"{trainable:,}")
print("Trainable percent:", round(100 * trainable / total, 2), "%")


In [ ]:
def train_one_epoch(optimizer, scheduler, epoch, use_aux=False):
    model.train()

    running = {
        "loss": 0.0,
        "cls_loss": 0.0,
        "contrastive_loss": 0.0,
        "consistency_loss": 0.0,
    }

    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(train_loader, desc=f"Train epoch {epoch}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        x = move_batch_to_device(batch)

        with amp_context():
            outputs = model(
                input_ids=x["input_ids"],
                attention_mask=x["attention_mask"],
                pixel_values=x["pixel_values"],
            )

            loss, logs = compute_loss(
                outputs,
                x["labels"],
                use_aux=use_aux,
            )

            loss = loss / GRAD_ACCUM

        loss.backward()

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=1.0,
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        for k in running:
            running[k] += logs[k]

        avg_loss = running["loss"] / step
        pbar.set_postfix({
            "loss": f"{avg_loss:.4f}",
            "cls": f"{running['cls_loss'] / step:.4f}",
        })

    return {k: v / len(train_loader) for k, v in running.items()}


@torch.no_grad()
def evaluate_model(save_rows=True):
    model.eval()

    all_true = []
    all_pred = []
    rows = []

    total_loss = 0.0
    total_batches = 0

    pbar = tqdm(val_loader, desc="Validation", leave=True)

    for batch in pbar:
        x = move_batch_to_device(batch)

        with amp_context():
            outputs = model(
                input_ids=x["input_ids"],
                attention_mask=x["attention_mask"],
                pixel_values=x["pixel_values"],
            )

            loss, _ = compute_loss(
                outputs,
                x["labels"],
                use_aux=False,
            )

        probs = torch.softmax(outputs["logits"], dim=1)
        preds = probs.argmax(dim=1)

        labels_cpu = x["labels"].detach().cpu().numpy()
        preds_cpu = preds.detach().cpu().numpy()
        probs_cpu = probs.detach().float().cpu().numpy()

        all_true.extend(labels_cpu.tolist())
        all_pred.extend(preds_cpu.tolist())

        total_loss += float(loss.detach().cpu())
        total_batches += 1

        if save_rows:
            sim_cpu = outputs["sim_score"].detach().float().cpu().numpy()
            amb_cpu = outputs["ambiguity"].detach().float().cpu().numpy()
            alpha_cpu = outputs["alpha"].detach().float().cpu().numpy()

            for i in range(len(preds_cpu)):
                row = {
                    "submission_id": batch["submission_id"][i],
                    "text": batch["text"][i],
                    "image_path": batch["image_path"][i],
                    "true_label_id": int(labels_cpu[i]),
                    "true_label": CLASS_NAMES[int(labels_cpu[i])],
                    "pred_label_id": int(preds_cpu[i]),
                    "pred_label": CLASS_NAMES[int(preds_cpu[i])],
                    "sim_score": float(sim_cpu[i]),
                    "ambiguity": float(amb_cpu[i]),
                    "gate_alpha": float(alpha_cpu[i]),
                }

                for j, name in enumerate(CLASS_NAMES):
                    row[f"prob_{name.lower()}"] = float(probs_cpu[i, j])

                rows.append(row)

    metrics = {
        "val_loss": total_loss / max(1, total_batches),
        "accuracy": accuracy_score(all_true, all_pred),
        "macro_f1": f1_score(all_true, all_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(all_true, all_pred, average="weighted", zero_division=0),
    }

    pred_df = pd.DataFrame(rows) if save_rows else pd.DataFrame()

    return metrics, pred_df


def save_checkpoint(path, epoch, metrics, stage):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "stage": stage,
            "metrics": metrics,
            "model_name": MODEL_NAME,
            "class_names": CLASS_NAMES,
            "text_col": TEXT_COL,
            "label_col": LABEL_COL,
        },
        path,
    )


def save_eval_artifacts(metrics, pred_df, prefix="val"):
    metrics_path = OUT_DIR / f"{prefix}_metrics.json"
    report_path = OUT_DIR / f"{prefix}_classification_report.txt"
    pred_path = OUT_DIR / f"{prefix}_predictions.csv"
    cm_path = OUT_DIR / f"{prefix}_confusion_matrix.png"

    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)

    report = classification_report(
        pred_df["true_label_id"],
        pred_df["pred_label_id"],
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        zero_division=0,
    )

    with open(report_path, "w") as f:
        f.write(report)

    pred_df.to_csv(pred_path, index=False)

    cm = confusion_matrix(
        pred_df["true_label_id"],
        pred_df["pred_label_id"],
        labels=list(range(NUM_CLASSES)),
    )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{prefix} confusion matrix")
    plt.tight_layout()
    plt.savefig(cm_path, dpi=200)
    plt.show()

    print("Saved:", metrics_path)
    print("Saved:", report_path)
    print("Saved:", pred_path)
    print("Saved:", cm_path)


print("Training and evaluation helpers ready")


In [ ]:
model.eval()

batch = next(iter(train_loader))
x = move_batch_to_device(batch)

with torch.no_grad():
    with amp_context():
        outputs = model(
            input_ids=x["input_ids"],
            attention_mask=x["attention_mask"],
            pixel_values=x["pixel_values"],
        )

print("logits:", outputs["logits"].shape)
print("f_text:", outputs["f_text"].shape)
print("f_image:", outputs["f_image"].shape)
print("f_cross:", outputs["f_cross"].shape)
print("sim_score:", outputs["sim_score"].shape)
print("ambiguity:", outputs["ambiguity"].shape)
print("alpha:", outputs["alpha"].shape)

loss, logs = compute_loss(outputs, x["labels"], use_aux=False)
print("loss:", float(loss.detach().cpu()))
print("logs:", logs)

probs = torch.softmax(outputs["logits"], dim=1)
print("probs shape:", probs.shape)
print("first pred:", CLASS_NAMES[int(probs[0].argmax().detach().cpu())])
print("first true:", CLASS_NAMES[int(x["labels"][0].detach().cpu())])


In [ ]:
# Stage 1: freeze CLIP, train only CMCN fusion + classifier.
set_clip_trainable(unfreeze_last_layers=0)

total, trainable = count_params()
print("Stage 1: frozen CLIP baseline")
print("Trainable parameters:", f"{trainable:,}", "/", f"{total:,}")

optimizer, scheduler = make_optimizer_and_scheduler(
    epochs=EPOCHS_STAGE1,
    head_lr=LR_HEAD,
    clip_lr=0.0,
)

history = []
best_macro_f1 = -1.0
best_path = OUT_DIR / "best_cmcn.pt"
last_path = OUT_DIR / "last_cmcn.pt"

for epoch in range(1, EPOCHS_STAGE1 + 1):
    start = time.time()

    train_metrics = train_one_epoch(
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch,
        use_aux=False,
    )

    val_metrics, _ = evaluate_model(save_rows=False)

    elapsed_min = (time.time() - start) / 60

    row = {
        "epoch": epoch,
        "stage": "stage1_frozen_clip",
        "elapsed_min": elapsed_min,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

    history.append(row)
    pd.DataFrame(history).to_csv(OUT_DIR / "history.csv", index=False)

    print(json.dumps(row, indent=2))

    save_checkpoint(
        path=last_path,
        epoch=epoch,
        metrics=val_metrics,
        stage="stage1_frozen_clip",
    )

    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]
        save_checkpoint(
            path=best_path,
            epoch=epoch,
            metrics=val_metrics,
            stage="stage1_frozen_clip",
        )
        print("New best macro-F1:", best_macro_f1)
        print("Saved:", best_path)

print("Stage 1 done")
print("Best macro-F1:", best_macro_f1)


In [ ]:
# Recover from epoch-2 NaN collapse.
best_path = OUT_DIR / "best_cmcn.pt"

ckpt = torch.load(best_path, map_location=device, weights_only=False)

model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

print("Reloaded best checkpoint")
print("Checkpoint epoch:", ckpt["epoch"])
print("Checkpoint stage:", ckpt["stage"])
print("Checkpoint metrics:", ckpt["metrics"])

# Lower LR to avoid classifier blow-up.
LR_HEAD_SAFE = 5e-5

def compute_loss(outputs, labels, use_aux=False):
    # Compute CE in float32 for numerical stability, even under bf16 autocast.
    logits = outputs["logits"].float()
    weight = class_weights.to(device=logits.device, dtype=torch.float32)

    cls_loss = F.cross_entropy(
        logits,
        labels,
        weight=weight,
    )

    contrastive_loss, consistency_loss = cmcn_aux_losses(outputs, labels)

    if use_aux:
        total_loss = cls_loss + 0.3 * contrastive_loss.float() + 0.1 * consistency_loss.float()
    else:
        total_loss = cls_loss

    logs = {
        "loss": float(total_loss.detach().float().cpu()),
        "cls_loss": float(cls_loss.detach().float().cpu()),
        "contrastive_loss": float(contrastive_loss.detach().float().cpu()),
        "consistency_loss": float(consistency_loss.detach().float().cpu()),
    }

    return total_loss, logs


def train_one_epoch_safe(optimizer, scheduler, epoch, use_aux=False):
    model.train()

    running = {
        "loss": 0.0,
        "cls_loss": 0.0,
        "contrastive_loss": 0.0,
        "consistency_loss": 0.0,
    }

    optimizer.zero_grad(set_to_none=True)
    valid_steps = 0

    pbar = tqdm(train_loader, desc=f"Train epoch {epoch}", leave=True)

    for step, batch in enumerate(pbar, start=1):
        x = move_batch_to_device(batch)

        with amp_context():
            outputs = model(
                input_ids=x["input_ids"],
                attention_mask=x["attention_mask"],
                pixel_values=x["pixel_values"],
            )
            loss, logs = compute_loss(outputs, x["labels"], use_aux=use_aux)

        if not torch.isfinite(loss):
            print(f"Skipping non-finite loss at step {step}:", loss.item())
            optimizer.zero_grad(set_to_none=True)
            continue

        (loss / GRAD_ACCUM).backward()

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=0.5,
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        for k in running:
            running[k] += logs[k]

        valid_steps += 1
        pbar.set_postfix({
            "loss": f"{running['loss'] / max(1, valid_steps):.4f}",
            "cls": f"{running['cls_loss'] / max(1, valid_steps):.4f}",
        })

    return {k: v / max(1, valid_steps) for k, v in running.items()}


print("Patched compute_loss + safe trainer")
print("Use LR_HEAD_SAFE:", LR_HEAD_SAFE)


In [ ]:
metrics, pred_df = evaluate_model(save_rows=True)

print("Final validation metrics:")
print(json.dumps(metrics, indent=2))

save_eval_artifacts(
    metrics=metrics,
    pred_df=pred_df,
    prefix="val_best_epoch1"
)

display(pred_df.head())


In [ ]:
# Continue safely from best epoch-1 checkpoint.
set_clip_trainable(unfreeze_last_layers=0)

optimizer, scheduler = make_optimizer_and_scheduler(
    epochs=2,
    head_lr=LR_HEAD_SAFE,
    clip_lr=0.0,
)

safe_history = []

for epoch in [2, 3]:
    start = time.time()

    train_metrics = train_one_epoch_safe(
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch,
        use_aux=False,
    )

    val_metrics, _ = evaluate_model(save_rows=False)
    elapsed_min = (time.time() - start) / 60

    row = {
        "epoch": epoch,
        "stage": "stage1_safe_resume",
        "elapsed_min": elapsed_min,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

    safe_history.append(row)
    print(json.dumps(row, indent=2))

    save_checkpoint(
        path=OUT_DIR / "last_cmcn_safe.pt",
        epoch=epoch,
        metrics=val_metrics,
        stage="stage1_safe_resume",
    )

    if val_metrics["macro_f1"] > ckpt["metrics"]["macro_f1"]:
        save_checkpoint(
            path=OUT_DIR / "best_cmcn.pt",
            epoch=epoch,
            metrics=val_metrics,
            stage="stage1_safe_resume",
        )
        print("New safe best saved:", OUT_DIR / "best_cmcn.pt")


In [ ]:
# Save current epoch-3 model as current best before Stage 2.
save_checkpoint(
    path=OUT_DIR / "best_cmcn.pt",
    epoch=3,
    metrics={
        "val_loss": 0.7611598936489786,
        "accuracy": 0.8075,
        "macro_f1": 0.7225111393912291,
        "weighted_f1": 0.8122644454048068,
    },
    stage="stage1_safe_resume",
)

print("Saved epoch 3 as current best checkpoint")

# Stage 2: CMCN auxiliary-loss training for epochs 4 and 5.
# CLIP remains frozen.
# Loss = CE + 0.3 * contrastive + 0.1 * consistency

set_clip_trainable(unfreeze_last_layers=0)

optimizer, scheduler = make_optimizer_and_scheduler(
    epochs=2,
    head_lr=LR_HEAD_SAFE * 0.5,
    clip_lr=0.0,
)

current_best_f1 = 0.7225111393912291
stage2_history = []

for epoch in [4, 5]:
    start = time.time()

    train_metrics = train_one_epoch_safe(
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch,
        use_aux=True,
    )

    val_metrics, _ = evaluate_model(save_rows=False)
    elapsed_min = (time.time() - start) / 60

    row = {
        "epoch": epoch,
        "stage": "stage2_cmcn_aux_losses",
        "elapsed_min": elapsed_min,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

    stage2_history.append(row)
    print(json.dumps(row, indent=2))

    save_checkpoint(
        path=OUT_DIR / "last_cmcn_stage2.pt",
        epoch=epoch,
        metrics=val_metrics,
        stage="stage2_cmcn_aux_losses",
    )

    if val_metrics["macro_f1"] > current_best_f1:
        current_best_f1 = val_metrics["macro_f1"]

        save_checkpoint(
            path=OUT_DIR / "best_cmcn.pt",
            epoch=epoch,
            metrics=val_metrics,
            stage="stage2_cmcn_aux_losses",
        )

        print("New best checkpoint saved")
        print("Best macro-F1:", current_best_f1)

pd.DataFrame(stage2_history).to_csv(OUT_DIR / "stage2_history.csv", index=False)

print("Stage 2 complete")
print("Best macro-F1:", current_best_f1)
print("Best checkpoint:", OUT_DIR / "best_cmcn.pt")


In [ ]:
# Stage 3: unfreeze last CLIP text + vision layer for one careful epoch.
# Goal: small accuracy boost without destroying CLIP features.

ckpt = torch.load(OUT_DIR / "best_cmcn.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

print("Loaded best before Stage 3")
print("Best checkpoint epoch:", ckpt["epoch"])
print("Best metrics:", ckpt["metrics"])

set_clip_trainable(
    unfreeze_last_layers=1,
    train_clip_projection=True,
)

total, trainable = count_params()
print("Stage 3 trainable parameters:", f"{trainable:,}", "/", f"{total:,}")

optimizer, scheduler = make_optimizer_and_scheduler(
    epochs=1,
    head_lr=LR_HEAD_SAFE * 0.25,
    clip_lr=1e-6,
)

current_best_f1 = ckpt["metrics"]["macro_f1"]

start = time.time()

train_metrics = train_one_epoch_safe(
    optimizer=optimizer,
    scheduler=scheduler,
    epoch=6,
    use_aux=True,
)

val_metrics, _ = evaluate_model(save_rows=False)
elapsed_min = (time.time() - start) / 60

row = {
    "epoch": 6,
    "stage": "stage3_unfreeze_last_clip_layer",
    "elapsed_min": elapsed_min,
    **{f"train_{k}": v for k, v in train_metrics.items()},
    **{f"val_{k}": v for k, v in val_metrics.items()},
}

print(json.dumps(row, indent=2))

save_checkpoint(
    path=OUT_DIR / "last_cmcn_stage3.pt",
    epoch=6,
    metrics=val_metrics,
    stage="stage3_unfreeze_last_clip_layer",
)

if val_metrics["macro_f1"] > current_best_f1:
    save_checkpoint(
        path=OUT_DIR / "best_cmcn.pt",
        epoch=6,
        metrics=val_metrics,
        stage="stage3_unfreeze_last_clip_layer",
    )
    print("New best checkpoint saved")
else:
    print("Stage 3 did not improve macro-F1. Keeping previous best checkpoint.")

print("Best checkpoint:", OUT_DIR / "best_cmcn.pt")


In [ ]:
# Stage 3 continuation: one more careful epoch with lower LR.

ckpt = torch.load(OUT_DIR / "best_cmcn.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

print("Loaded best checkpoint before epoch 7")
print("Best checkpoint epoch:", ckpt["epoch"])
print("Best metrics:", ckpt["metrics"])

set_clip_trainable(
    unfreeze_last_layers=1,
    train_clip_projection=True,
)

optimizer, scheduler = make_optimizer_and_scheduler(
    epochs=1,
    head_lr=LR_HEAD_SAFE * 0.10,
    clip_lr=5e-7,
)

current_best_f1 = ckpt["metrics"]["macro_f1"]

start = time.time()

train_metrics = train_one_epoch_safe(
    optimizer=optimizer,
    scheduler=scheduler,
    epoch=7,
    use_aux=True,
)

val_metrics, _ = evaluate_model(save_rows=False)
elapsed_min = (time.time() - start) / 60

row = {
    "epoch": 7,
    "stage": "stage3_unfreeze_last_clip_layer_lower_lr",
    "elapsed_min": elapsed_min,
    **{f"train_{k}": v for k, v in train_metrics.items()},
    **{f"val_{k}": v for k, v in val_metrics.items()},
}

print(json.dumps(row, indent=2))

save_checkpoint(
    path=OUT_DIR / "last_cmcn_stage3_epoch7.pt",
    epoch=7,
    metrics=val_metrics,
    stage="stage3_unfreeze_last_clip_layer_lower_lr",
)

if val_metrics["macro_f1"] > current_best_f1:
    save_checkpoint(
        path=OUT_DIR / "best_cmcn.pt",
        epoch=7,
        metrics=val_metrics,
        stage="stage3_unfreeze_last_clip_layer_lower_lr",
    )
    print("New best checkpoint saved")
else:
    print("Epoch 7 did not improve macro-F1. Keeping previous best checkpoint.")

print("Best checkpoint:", OUT_DIR / "best_cmcn.pt")


In [ ]:
save_checkpoint(
    path=OUT_DIR / "best_cmcn.pt",
    epoch=7,
    metrics={
        "val_loss": 0.9039539637418982,
        "accuracy": 0.8433,
        "macro_f1": 0.762221039726063,
        "weighted_f1": 0.8463643390857365,
    },
    stage="stage3_unfreeze_last_clip_layer_lower_lr",
)

print("Saved epoch 7 as best checkpoint")


In [ ]:
# Epoch 8: final careful continuation with same 1-layer CLIP unfreeze.

ckpt = torch.load(OUT_DIR / "best_cmcn.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

set_clip_trainable(
    unfreeze_last_layers=1,
    train_clip_projection=True,
)

optimizer, scheduler = make_optimizer_and_scheduler(
    epochs=1,
    head_lr=LR_HEAD_SAFE * 0.075,
    clip_lr=3e-7,
)

current_best_f1 = ckpt["metrics"]["macro_f1"]

start = time.time()

train_metrics = train_one_epoch_safe(
    optimizer=optimizer,
    scheduler=scheduler,
    epoch=8,
    use_aux=True,
)

val_metrics, _ = evaluate_model(save_rows=False)
elapsed_min = (time.time() - start) / 60

row = {
    "epoch": 8,
    "stage": "stage3_final_careful_epoch",
    "elapsed_min": elapsed_min,
    **{f"train_{k}": v for k, v in train_metrics.items()},
    **{f"val_{k}": v for k, v in val_metrics.items()},
}

print(json.dumps(row, indent=2))

save_checkpoint(
    path=OUT_DIR / "last_cmcn_epoch8.pt",
    epoch=8,
    metrics=val_metrics,
    stage="stage3_final_careful_epoch",
)

if val_metrics["macro_f1"] > current_best_f1:
    save_checkpoint(
        path=OUT_DIR / "best_cmcn.pt",
        epoch=8,
        metrics=val_metrics,
        stage="stage3_final_careful_epoch",
    )
    print("New best checkpoint saved")
else:
    print("Epoch 8 did not improve macro-F1. Keeping epoch 7 best.")


In [ ]:
save_checkpoint(
    path=OUT_DIR / "best_cmcn.pt",
    epoch=8,
    metrics={
        "val_loss": 0.9135795089240653,
        "accuracy": 0.8438,
        "macro_f1": 0.7634019206914916,
        "weighted_f1": 0.8465937092267275,
    },
    stage="stage3_final_careful_epoch",
)

print("Saved epoch 8 as best checkpoint")


In [ ]:
ckpt = torch.load(OUT_DIR / "best_cmcn.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()

print("Loaded best checkpoint")
print("Best epoch:", ckpt["epoch"])
print("Best stage:", ckpt["stage"])
print("Best metrics:", ckpt["metrics"])

metrics, pred_df = evaluate_model(save_rows=True)

print("Final metrics:")
print(json.dumps(metrics, indent=2))

save_eval_artifacts(
    metrics=metrics,
    pred_df=pred_df,
    prefix="final_best"
)

display(pred_df.head())


In [ ]:
@torch.no_grad()
def evaluate_with_logit_bias(misleading_bias=-0.15):
    model.eval()

    all_true = []
    all_pred = []

    for batch in tqdm(val_loader, desc="calibrated eval"):
        x = move_batch_to_device(batch)

        with amp_context():
            outputs = model(
                input_ids=x["input_ids"],
                attention_mask=x["attention_mask"],
                pixel_values=x["pixel_values"],
            )

        logits = outputs["logits"].float()

        # Class 2 = Misleading
        logits[:, 2] += misleading_bias

        preds = logits.argmax(dim=1)

        all_true.extend(x["labels"].cpu().tolist())
        all_pred.extend(preds.cpu().tolist())

    return {
        "accuracy": accuracy_score(all_true, all_pred),
        "macro_f1": f1_score(all_true, all_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(all_true, all_pred, average="weighted", zero_division=0),
        "report": classification_report(
            all_true,
            all_pred,
            labels=list(range(NUM_CLASSES)),
            target_names=CLASS_NAMES,
            zero_division=0,
        ),
        "cm": confusion_matrix(all_true, all_pred, labels=list(range(NUM_CLASSES))),
    }

for bias in [-0.05, -0.10, -0.15, -0.20, -0.25, -0.30]:
    result = evaluate_with_logit_bias(misleading_bias=bias)
    print("bias:", bias, "accuracy:", result["accuracy"], "macro_f1:", result["macro_f1"])


In [ ]:
MISLEADING_BIAS = -0.30

@torch.no_grad()
def evaluate_calibrated_and_save(prefix="final_best_calibrated"):
    model.eval()

    all_true = []
    all_pred = []
    rows = []

    for batch in tqdm(val_loader, desc="calibrated final eval"):
        x = move_batch_to_device(batch)

        with amp_context():
            outputs = model(
                input_ids=x["input_ids"],
                attention_mask=x["attention_mask"],
                pixel_values=x["pixel_values"],
            )

        logits = outputs["logits"].float()
        logits[:, 2] += MISLEADING_BIAS

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        labels_cpu = x["labels"].detach().cpu().numpy()
        preds_cpu = preds.detach().cpu().numpy()
        probs_cpu = probs.detach().cpu().numpy()

        all_true.extend(labels_cpu.tolist())
        all_pred.extend(preds_cpu.tolist())

        sim_cpu = outputs["sim_score"].detach().float().cpu().numpy()
        amb_cpu = outputs["ambiguity"].detach().float().cpu().numpy()
        alpha_cpu = outputs["alpha"].detach().float().cpu().numpy()

        for i in range(len(preds_cpu)):
            row = {
                "submission_id": batch["submission_id"][i],
                "text": batch["text"][i],
                "image_path": batch["image_path"][i],
                "true_label_id": int(labels_cpu[i]),
                "true_label": CLASS_NAMES[int(labels_cpu[i])],
                "pred_label_id": int(preds_cpu[i]),
                "pred_label": CLASS_NAMES[int(preds_cpu[i])],
                "misleading_logit_bias": MISLEADING_BIAS,
                "sim_score": float(sim_cpu[i]),
                "ambiguity": float(amb_cpu[i]),
                "gate_alpha": float(alpha_cpu[i]),
            }

            for j, name in enumerate(CLASS_NAMES):
                row[f"prob_{name.lower()}"] = float(probs_cpu[i, j])

            rows.append(row)

    pred_df = pd.DataFrame(rows)

    metrics = {
        "accuracy": accuracy_score(all_true, all_pred),
        "macro_f1": f1_score(all_true, all_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(all_true, all_pred, average="weighted", zero_division=0),
        "misleading_logit_bias": MISLEADING_BIAS,
    }

    print(json.dumps(metrics, indent=2))
    print(
        classification_report(
            all_true,
            all_pred,
            labels=list(range(NUM_CLASSES)),
            target_names=CLASS_NAMES,
            zero_division=0,
        )
    )

    save_eval_artifacts(metrics, pred_df, prefix=prefix)

    return metrics, pred_df


cal_metrics, cal_pred_df = evaluate_calibrated_and_save()
display(cal_pred_df.head())


In [ ]:
@torch.no_grad()
def predict_one_calibrated(text, image_path):
    model.eval()

    enc = tokenizer(
        str(text),
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )

    image = Image.open(image_path).convert("RGB")
    pixels = image_processor(images=image, return_tensors="pt")["pixel_values"]

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)
    pixel_values = pixels.to(device)

    with amp_context():
        outputs = model(input_ids, attention_mask, pixel_values)

    logits = outputs["logits"].float()
    logits[:, 2] += MISLEADING_BIAS

    probs = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()
    pred_id = int(probs.argmax())

    return {
        "pred_label_id": pred_id,
        "pred_label": CLASS_NAMES[pred_id],
        "misleading_logit_bias": MISLEADING_BIAS,
        "probabilities": {CLASS_NAMES[i]: float(probs[i]) for i in range(NUM_CLASSES)},
        "sim_score": float(outputs["sim_score"][0].detach().float().cpu()),
        "ambiguity": float(outputs["ambiguity"][0].detach().float().cpu()),
        "gate_alpha": float(outputs["alpha"][0].detach().float().cpu()),
    }


In [ ]:
# Load best CMCN checkpoint for inference
ckpt = torch.load(
    OUT_DIR / "best_cmcn.pt",
    map_location=device,
    weights_only=False,
)

model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()

MISLEADING_BIAS = -0.30

print("Loaded checkpoint")
print("Epoch:", ckpt["epoch"])
print("Stage:", ckpt["stage"])
print("Metrics:", ckpt["metrics"])
print("Using misleading logit bias:", MISLEADING_BIAS)


@torch.no_grad()
def predict_one_calibrated(text, image_path):
    model.eval()

    enc = tokenizer(
        str(text),
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )

    image = Image.open(image_path).convert("RGB")
    pixels = image_processor(
        images=image,
        return_tensors="pt",
    )["pixel_values"]

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)
    pixel_values = pixels.to(device)

    with amp_context():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
        )

    logits = outputs["logits"].float()

    # Class 2 = Misleading
    logits[:, 2] += MISLEADING_BIAS

    probs = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()
    pred_id = int(probs.argmax())

    return {
        "pred_label_id": pred_id,
        "pred_label": CLASS_NAMES[pred_id],
        "misleading_logit_bias": MISLEADING_BIAS,
        "probabilities": {
            CLASS_NAMES[i]: float(probs[i])
            for i in range(NUM_CLASSES)
        },
        "sim_score": float(outputs["sim_score"][0].detach().float().cpu()),
        "ambiguity": float(outputs["ambiguity"][0].detach().float().cpu()),
        "gate_alpha": float(outputs["alpha"][0].detach().float().cpu()),
    }


# Test on one validation sample
sample = val_df.iloc[0]

result = predict_one_calibrated(
    text=sample[TEXT_COL],
    image_path=sample["image_path"],
)

print("Text:", sample[TEXT_COL])
print("Image:", sample["image_path"])
print("True label:", CLASS_NAMES[int(sample[LABEL_COL])])
print(json.dumps(result, indent=2))


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

# Pick 2 random validation examples.
examples = val_df.sample(2, random_state=7).reset_index(drop=True)

for i, row in examples.iterrows():
    result = predict_one_calibrated(
        text=row[TEXT_COL],
        image_path=row["image_path"],
    )

    img = Image.open(row["image_path"]).convert("RGB")

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title(
        f"Title: {row[TEXT_COL]}\n"
        f"True: {CLASS_NAMES[int(row[LABEL_COL])]}\n"
        f"Pred: {result['pred_label']} "
        f"({max(result['probabilities'].values()):.2%})",
        fontsize=10,
    )
    plt.show()

    print("Submission ID:", row["submission_id"])
    print("Title:", row[TEXT_COL])
    print("Image:", row["image_path"])
    print("True label:", CLASS_NAMES[int(row[LABEL_COL])])
    print("Predicted label:", result["pred_label"])
    print("Probabilities:")
    for label, prob in result["probabilities"].items():
        print(f"  {label}: {prob:.4f}")
    print("sim_score:", result["sim_score"])
    print("ambiguity:", result["ambiguity"])
    print("gate_alpha:", result["gate_alpha"])
    print("-" * 80)


In [ ]:
# Example: one True and one Manipulated


import matplotlib.pyplot as plt
from IPython.display import display

# Pick 2 random validation examples.
examples = pd.concat([
    val_df[val_df[LABEL_COL] == 0].sample(1, random_state=11),
    val_df[val_df[LABEL_COL] == 4].sample(1, random_state=12),
]).reset_index(drop=True)

for i, row in examples.iterrows():
    result = predict_one_calibrated(
        text=row[TEXT_COL],
        image_path=row["image_path"],
    )

    img = Image.open(row["image_path"]).convert("RGB")

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title(
        f"Title: {row[TEXT_COL]}\n"
        f"True: {CLASS_NAMES[int(row[LABEL_COL])]}\n"
        f"Pred: {result['pred_label']} "
        f"({max(result['probabilities'].values()):.2%})",
        fontsize=10,
    )
    plt.show()

    print("Submission ID:", row["submission_id"])
    print("Title:", row[TEXT_COL])
    print("Image:", row["image_path"])
    print("True label:", CLASS_NAMES[int(row[LABEL_COL])])
    print("Predicted label:", result["pred_label"])
    print("Probabilities:")
    for label, prob in result["probabilities"].items():
        print(f"  {label}: {prob:.4f}")
    print("sim_score:", result["sim_score"])
    print("ambiguity:", result["ambiguity"])
    print("gate_alpha:", result["gate_alpha"])
    print("-" * 80)

